In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader,DirectoryLoader

c:\Users\biren\anaconda3\envs\constitution\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_pdf(data):
    loader = PyPDFLoader(data)
    documents = loader.load()
    return documents    

In [4]:
docs = load_pdf("../data/the_constitution_of_india.pdf")

In [5]:
len(docs)

402

In [6]:
docs[10]

Document(metadata={'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2024-07-01T11:20:33+00:00', 'source': '../data/the_constitution_of_india.pdf', 'total_pages': 402, 'page': 10, 'page_label': '11'}, page_content='ContentsARTICLES(viii)115.Supplementary, additional or excess grants.116. Votes on account, votes of credit and exceptional grants. 117. Special provisions as to financial Bills. Procedure Generally 118.Rules of procedure. 119. Regulation by law of procedure in Parliament in relation to financial business.120.Language to be used in Parliament.121.Restriction on discussion in Parliament.122. Courts not to inquire into proceedings of Parliament. CHAPTER III.LEGISLATIVE  POWERS OF THE PRESIDENT123.Power of President to promulgate Ordinances during recess of  Parliament.CHAPTER IV.THE UNION JUDICIARY124.Establishment and constitution ofthe Supreme Court.124A.National Judicial AppointmentsCommission.124B.Functions of Commission.124C.Power of Parliament t

In [7]:
from typing import List
from langchain.schema import Document

In [8]:
def filter_document(docs:List[Document])-> List[Document]:
    filtered_docs = []
    for doc in docs:
        document = Document(
            page_content=doc.page_content,
            metadata={"source": doc.metadata.get("source","Unknown")}
        )
        filtered_docs.append(document)
    return filtered_docs

In [9]:
filtered_docs = filter_document(docs)

In [10]:
filtered_docs[10]

Document(metadata={'source': '../data/the_constitution_of_india.pdf'}, page_content='ContentsARTICLES(viii)115.Supplementary, additional or excess grants.116. Votes on account, votes of credit and exceptional grants. 117. Special provisions as to financial Bills. Procedure Generally 118.Rules of procedure. 119. Regulation by law of procedure in Parliament in relation to financial business.120.Language to be used in Parliament.121.Restriction on discussion in Parliament.122. Courts not to inquire into proceedings of Parliament. CHAPTER III.LEGISLATIVE  POWERS OF THE PRESIDENT123.Power of President to promulgate Ordinances during recess of  Parliament.CHAPTER IV.THE UNION JUDICIARY124.Establishment and constitution ofthe Supreme Court.124A.National Judicial AppointmentsCommission.124B.Functions of Commission.124C.Power of Parliament to make law.125.Salaries, etc., of Judges.126.Appointment of acting Chief Justice.127.Appointment ofad hocJudges.128.Attendance of retired Judges at sittings

text chunks

In [11]:
def text_splitter(documents: List[Document]) -> List[Document]:
    """ Given a list of documents objects, return a list of documents with only the page content and source metadata. """
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunk = text_splitter.split_documents(documents)
    return text_chunk

In [12]:
text_chunks = text_splitter(filtered_docs)   
print("length of text chunks: ", len(text_chunks))

length of text chunks:  2006


In [13]:
from langchain.embeddings import HuggingFaceEmbeddings

def get_embeddings_vector():
    """Load and return a HuggingFace embedding model."""
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

embeddings_model = get_embeddings_vector()

C:\Users\biren\AppData\Local\Temp\ipykernel_6996\1429207267.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)


### load environments variables

In [35]:

import os

from dotenv import load_dotenv
load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
# os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [22]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY
pinecone = Pinecone(api_key=pinecone_api_key)


#### create index

In [23]:
from pinecone import ServerlessSpec

index_name = "constitution-bot"
serverless_spec = ServerlessSpec(cloud="aws", region="us-east-1")
if not pinecone.has_index(index_name):
    pinecone.create_index(index_name, 
                          dimension=384, 
                          metric="cosine",
                          spec=serverless_spec)
index = pinecone.Index(index_name)

#### upsert vectors of pinecone

In [24]:
from langchain_pinecone import PineconeVectorStore
def create_vectorstore(index_name, model, text_chunks):
    vectorstore = PineconeVectorStore.from_documents(
        documents=text_chunks,
        embedding=model,
        index_name=index_name
    )
    return vectorstore

vectorstore = create_vectorstore(index_name, embeddings_model, text_chunks)

In [26]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
def fetch_vectorstore(index_name, embedding):   
    docsearch = PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embedding
    )
    return docsearch

In [27]:
vectorstore = fetch_vectorstore(index_name, embeddings_model)

#### retrieveal from documents/vector

In [28]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [29]:
retrieved_docs = retriever.invoke("schedule right articles")
retrieved_docs

[Document(id='2226dc99-8ecf-4187-aa88-e051997dbd1f', metadata={'source': '../data/the_constitution_of_india.pdf'}, page_content='Laws made under articles 2 and 3 to provide for the amendment of the First and the Fourth Schedules and supplemental, incidental and consequential matters.—(1) Any law referred to in article 2 or article 3 shall contain such provisions for the amendment of the First Schedule and the Fourth Schedule as may be necessary to give effect to the provisions of the law and may also contain such supplemental, incidental and consequential provisions (including provisions as to representation in'),
 Document(id='dd08548f-efa3-465d-8ca4-4f7c0b66463c', metadata={'source': '../data/the_constitution_of_india.pdf'}, page_content='the Schedule is so amended, any reference to this Schedule in this Constitution shall be construed as a reference to such Schedule as so amended.(2) No such law as is mentioned in sub-paragraph (1) of this paragraph shall be deemed to be an amendmen

#### LLM integration

In [36]:
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

In [37]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "You are an Constitutional assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [39]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [41]:
response = rag_chain.invoke({"input": "how to use schedule caste right"})
print(response["answer"])

To use Scheduled Caste rights, individuals must first ensure they are registered as a Scheduled Caste member. They can then access benefits such as education and job reservations, and protection under laws like the Scheduled Castes and Scheduled Tribes (Prevention of Atrocities) Act. Additionally, they can consult the National Commission for Scheduled Castes for guidance on major policy matters affecting their community.
